# FashionMNIST

![FashionMNIST](https://github.com/zalandoresearch/fashion-mnist/raw/master/doc/img/fashion-mnist-sprite.png)

Fashion-MNIST is a dataset of Zalando's article images—consisting of a training set of 60,000 examples and a test set of 10,000 examples. 

Each example is a 28x28 grayscale image, associated with a label from 10 classes.

Time to train our network!

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

# (Down)loading the Datasets

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26421880/26421880 [00:00<00:00, 83412160.90it/s]


Extracting data/FashionMNIST/raw/train-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 2193894.46it/s]

Extracting data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to data/FashionMNIST/raw




100%|██████████| 4422102/4422102 [00:00<00:00, 39724655.14it/s]


Extracting data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 12445116.42it/s]


Extracting data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to data/FashionMNIST/raw



In [3]:
len(training_data), len(test_data)

(60000, 10000)

In [4]:
10_000 / 64

156.25

# Preparing the Dataloaders

In [5]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in train_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## Creating Model

In [6]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc1 = nn.Linear(in_features=64*6*6, out_features=600)
        self.drop = nn.Dropout2d(0.25)
        self.fc2 = nn.Linear(in_features=600, out_features=120)
        self.fc3 = nn.Linear(in_features=120, out_features=10)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.fc3(out)
        
        return out

model = NeuralNetwork()
print(model)

Using cuda device
NeuralNetwork(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=2304, out_features=600, bias=True)
  (drop): Dropout2d(p=0.25, inplace=False)
  (fc2): Linear(in_features=600, out_features=120, bias=True)
  (fc3): Linear(in_features=120, out_features=10, bias=True)
)


# Define the optimizer

We will use the SGD optimizer, with a learning rate of 0.001

In [7]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

# Define the loss function

We will use the CrossEntropyLoss loss function, which is a loss function for classification tasks.

In [8]:
loss_fn = nn.CrossEntropyLoss()

# Send the model to the GPU


In [9]:
model.to(device)

NeuralNetwork(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=2304, out_features=600, bias=True)
  (drop): Dropout2d(p=0.25, inplace=False)
  (fc2): Linear(in_features=600, out_features=120, bias=True)
  (fc3): Linear(in_features=120, out_features=10, bias=True)
)

# Complete the training loop

In [10]:
epochs = 5
for t in range(epochs):
	print(f"Epoch {t+1}\n-------------------------------")
	size = len(train_dataloader.dataset)
	model.train()
	for batch, (X, y) in enumerate(train_dataloader):
		# Send data to the GPU
		X = X.to(device)
		y = y.to(device)

		# Compute prediction
		pred = model(X)
		
		# Compute loss
		loss = loss_fn(pred, y)

		# Backpropagation
		loss.backward()
		
		# Update weights
		optimizer.step()
		
		# Reset gradients
		optimizer.zero_grad()

		if batch % 100 == 0:
			loss, current = loss.item(), (batch + 1) * len(X)
			print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

Epoch 1
-------------------------------


/home/ezalos/42/Notebooks2Teach/venv/lib/python3.9/site-packages/torch/nn/functional.py:1347: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  warnings.warn(warn_msg)


loss: 2.294279  [   64/60000]
loss: 1.859656  [ 6464/60000]
loss: 1.408376  [12864/60000]
loss: 1.337141  [19264/60000]
loss: 1.060029  [25664/60000]
loss: 0.998688  [32064/60000]
loss: 0.903299  [38464/60000]
loss: 0.840589  [44864/60000]
loss: 0.847200  [51264/60000]
loss: 0.722625  [57664/60000]
Epoch 2
-------------------------------
loss: 0.710505  [   64/60000]
loss: 0.855085  [ 6464/60000]
loss: 0.513586  [12864/60000]
loss: 0.753918  [19264/60000]
loss: 0.692017  [25664/60000]
loss: 0.653175  [32064/60000]
loss: 0.636644  [38464/60000]
loss: 0.686029  [44864/60000]
loss: 0.642893  [51264/60000]
loss: 0.592177  [57664/60000]
Epoch 3
-------------------------------
loss: 0.502215  [   64/60000]
loss: 0.721525  [ 6464/60000]
loss: 0.426612  [12864/60000]
loss: 0.636877  [19264/60000]
loss: 0.606874  [25664/60000]
loss: 0.547977  [32064/60000]
loss: 0.544967  [38464/60000]
loss: 0.638890  [44864/60000]
loss: 0.595351  [51264/60000]
loss: 0.482750  [57664/60000]
Epoch 4
------------

In [11]:
size = len(test_dataloader.dataset)
num_batches = len(test_dataloader)
model.eval()

test_loss = 0
correct = 0
with torch.no_grad():
	for X, y in test_dataloader:
		# Send data to the GPU
		X = X.to(device)
		y = y.to(device)
		# Compute prediction
		pred = model(X)
		# Compute loss, and add it to the test loss
		test_loss += loss_fn(pred, y).item()
		# Compute the number of correct predictions, and add it to the correct count
		correct += (pred.argmax(1) == y).type(torch.float).sum().item()

test_loss /= num_batches
correct /= size

print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

Test Error: 
 Accuracy: 84.1%, Avg loss: 0.450517 



# Predicting 

In [13]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
DATA_IDX = 9
x, y = test_data[DATA_IDX][0], test_data[DATA_IDX][1]

with torch.no_grad():
    x = x.to(device).unsqueeze(0)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Sneaker", Actual: "Sneaker"
